In [2]:
import pandas as pd
import time
import sys
import json
sys.path.insert(0, "../../utils/")
import xgboost as xgb
import lightgbm as lgb
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate
from sklearn.metrics import make_scorer, matthews_corrcoef
from joblib import dump
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score
)

In [ ]:
MODELS = {
    "SVM": SVC,
    "KNN": KNeighborsClassifier,
    "LogisticRegression": LogisticRegression,
    "RandomForest": RandomForestClassifier,
    "AdaBoost": AdaBoostClassifier,
    "GradientBoosting": GradientBoostingClassifier,
    "XGBoost": xgb.XGBClassifier,
    "LGBM": lgb.LGBMClassifier,
}

: 

: 

In [ ]:
GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 500, 1000, 3000, 5000],
        #"criterion": ["gini", "entropy"],
        "min_samples_split": [2, 10, 20],
        "min_samples_leaf": [1, 4, 8],
        #"max_features": ["sqrt", "log2"],
        "max_depth": [10, 30, 50, None]
    },
    "AdaBoost": {
        "n_estimators": [50, 300, 500, 1000],
        "learning_rate": [0.01, 0.1, 1.0]
    },
    "GradientBoosting": {
        "n_estimators": [100, 500, 1000, 3000],
        "learning_rate": [0.05, 0.01, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0]
    },
    "XGBoost": {
        "n_estimators": [100, 300, 500, 1000],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.01, 0.1],
        "subsample": [0.8, 1.0],
    },
    "LGBM": {
        "n_estimators": [100, 300, 500, 1000],
        "max_depth": [3, 5, -1],
        "learning_rate": [0.01, 0.1],
        "num_leaves": [15, 31],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    "KNN": {
        "n_neighbors": list(range(3, 21, 2)),
        #"weights": ["uniform", "distance"],
        #"metric": ["euclidean", "manhattan"]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        #"kernel": ["linear", "rbf"],
        #"gamma": ["scale"]
    },
    "LogisticRegression": {
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        #"solver": ["liblinear"],
        #"penalty": ["l1", "l2"]
    }
}

: 

: 

In [ ]:
def undersampling(df_data, seed):
    #ten_percent=df_data[df_data["target"]==2].sample(frac=0.10, random_state=seed)
    X=df_data.drop('target', axis=1)
    y=df_data['target']  
    #Se definen los objetos para submuestrear
    X['index']=X.index 
    undersampler=RandomUnderSampler(sampling_strategy='not minority', random_state=seed)    

    #Se aplica el submuestreo
    X_res, y_res=undersampler.fit_resample(X, y)
    df_resampled=pd.concat([X_res,y_res], axis=1)

    index_res=X_res['index']
    df_resampled.drop('index', axis=1, inplace=True)

    mask=~X['index'].isin(index_res)
    excluded_data=df_data[mask.values]
    #data_independent= pd.concat([ten_percent, excluded_data], axis=0)
    data_independent= pd.concat([excluded_data], axis=0)
    data_independent.reset_index(drop=True, inplace=True)
    data_independent.to_csv("../../models/data/data_independent.csv", index=False)
    
    return df_resampled

: 

: 

In [ ]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

: 

: 

In [ ]:
def function_split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

: 

: 

In [ ]:
def metrics(model, predict_val, y_val, dataset, div, predict_proba=None):
    acc_value = accuracy_score(y_pred=predict_val, y_true=y_val) 
    recall_value = recall_score(y_pred=predict_val, y_true=y_val, average='weighted')
    precision_value = precision_score(y_pred=predict_val, y_true=y_val, average='weighted') 
    f1_value = f1_score(y_pred=predict_val, y_true=y_val, average='weighted')
    mcc_value = matthews_corrcoef(y_pred=predict_val, y_true=y_val)
    cm = confusion_matrix(y_pred=predict_val, y_true=y_val)
    cm_dict = pd.DataFrame(cm).to_dict()

    roc_auc_value = None
    if predict_proba is not None:
        try:
            roc_auc_value = roc_auc_score(y_val, predict_proba, multi_class='ovr', average='weighted')
        except Exception as e:
            print(f"Error computing ROC AUC: {e}")
            roc_auc_value = None

    df_metrics = pd.DataFrame([[dataset, model, div, acc_value, recall_value, precision_value, f1_value, mcc_value, roc_auc_value, cm_dict]],
                              columns=["dataset", "model", "sampling", "acc", "recall", "precision", "f1", "mcc", "roc_auc", "conf_matrix"])

    return df_metrics

: 

: 

In [ ]:
def cross_function(model, X_train, y_train, cv):
    scoring_metrics = {
        "accuracy": "accuracy",
        "recall": "recall_weighted",
        "precision": "precision_weighted",
        "f1": "f1_weighted",
        "roc_auc_ovr": make_scorer(roc_auc_score, response_method="predict_proba", multi_class="ovr"),
        "mcc": make_scorer(matthews_corrcoef)
    }

    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics, return_train_score=True)

    df_val = pd.DataFrame({metric: scores[f'test_{metric}'] for metric in scoring_metrics})
    df_val['fit_time'] = scores['fit_time']
    df_val['score_time'] = scores['score_time']
    df_val['Dataset'] = 'Validation'

    df_train = pd.DataFrame({metric: scores[f'train_{metric}'] for metric in scoring_metrics})
    df_train['fit_time'] = scores['fit_time']
    df_train['score_time'] = scores['score_time']
    df_train['Dataset'] = 'Train'

    results_metrics = pd.concat([df_val, df_train], ignore_index=True)

    return results_metrics

: 

: 

In [ ]:
def get_model(model_name, seed):
    model_cls = MODELS[model_name]
    if model_name == "KNN":
        model = model_cls(n_jobs=-1)
    elif model_name == "SVM":
        model = model_cls(probability=True, random_state=seed)
    elif model_name in ["RandomForest", "XGBoost", "LGBM"]:
        model = model_cls(random_state=seed, n_jobs=-1)
    else:
        model = model_cls(random_state=seed)
    return model

: 

: 

In [ ]:
def function_train(model, model_name, train, val, div, seed):
    '''
    # Separa los datos de entrenamiento y validación
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values
    '''

    # Combina los conjuntos de entrenamiento y validación para la validación cruzada
    train_all= pd.concat([train, val], ignore_index=True)
    target = train_all["target"]
    train_all.drop(columns="target", inplace=True)
    #df_combined = val.copy()

    print(f"Train {model_name} with seed {seed} and division {div}")    
    results = []

    #Valdación cruzada antes de la búsqueda de hiperparámetros
    #model.fit(train_all, target)
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    cv_scores= cross_function(model, train_all, target, cv).copy()
    cv_scores["model"] = model_name
    cv_scores["sampling"] = div
    results.append(cv_scores)
    
    #cv_scores.to_csv(f"../../models/data/metrics/{model_name}_{seed}_{div}_cross_val.csv", index=False)

    #dump(model, f"../../models/{model_name}_{seed}_{div}.joblib")
    
    '''
    #Se predice en el conjunto de entrenamiento y validación
    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)

    y_pred_val = model.predict(X_val)
    y_proba_val = model.predict_proba(X_val)
    
    #Se obtiene un conjunto combinado de datos con las predicciones
    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    #df_combined.to_csv(f"../../models/data/{model_name}_{seed}_{div}_predictions.csv", index=False)
    
    #Se guardan las métricas de entrenamiento y validación
    train_metrics = metrics(model_name, y_pred_train, y_train, "Train", div, predict_proba=y_proba_train)
    val_metrics = metrics(model_name, y_pred_val, y_val, "Validation", div, predict_proba=y_proba_val)
    results.append(train_metrics)
    results.append(val_metrics)
    '''

    all_results = pd.concat(results, ignore_index=True)
    #all_results.to_csv(f"../../metrics/{model_name}_{seed}_{div}_all_metrics.csv", index=False)

    return all_results

: 

: 

In [ ]:
def grid_function(model, model_name, train, val, seed, div):
    grid = GridSearchCV(estimator=model, param_grid=GRIDS[model_name], cv=10, scoring="f1_weighted", n_jobs=-1)
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()
    print(f"GridSearchCV {model_name} with seed {seed} and division {div}")
    # Se realiza la búsqueda de hiperparámetros
    start_fit = time.time()
    grid.fit(X_train, y_train)
    elapsed_fit = time.time() - start_fit

    # Se obtienen los mejores parámetros y el mejor modelo
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_score = grid.best_score_

    best_params = json.dumps(best_params, indent=4)
    with open(f"../../models/data/json/{model_name}_Grid_{seed}_{div}_best_params.json", "w") as f:
        f.write(best_params)

    print(f"Best estimators: {best_model} and best parameters found: {best_params}")

    dump(best_model, f"../../models/data/best/{model_name}_Grid_{seed}_{div}_best.joblib")

    # Se predice en el conjunto de entrenamiento y validación
    y_pred_train = best_model.predict(X_train)
    y_proba_train = best_model.predict_proba(X_train)

    y_pred_val = best_model.predict(X_val)
    y_proba_val = best_model.predict_proba(X_val)

    # Se obtiene un conjunto combinado de datos con las predicciones
    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    #df_combined.to_csv(f"../../models/data/{model_name}_Grid_{seed}_{div}_best_predictions.csv", index=False)

    # Se guardan las métricas de entrenamiento y validación
    start_metrics = time.time()
    train_metrics = metrics(model_name, y_pred_train, y_train, "Train", div, predict_proba=y_proba_train)
    val_metrics = metrics(model_name, y_pred_val, y_val, "Validation", div, predict_proba=y_proba_val)
    elapsed_metrics = time.time() - start_metrics
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    results['fit_time'] = elapsed_fit
    results['metrics_time'] = elapsed_metrics
    return results

: 

: 

In [ ]:
def main_train(df_data, seed, model_name, grid_search=False):
    all_metrics = []
    all_metrics_grid = []
    df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = function_split(df_data, seed)
    model=get_model(model_name, seed)
    all_metrics= pd.concat([function_train(model, model_name, df_train, df_val, "Original", seed),
                            function_train(model, model_name, df_train_under, df_val_under, "Under", seed),
                            function_train(model, model_name, df_train_over, df_val_over, "Over", seed)], 
                            ignore_index=True)
    all_metrics.to_csv(f"../../metrics/{model_name}_{seed}_metrics.csv", index=False)
    if grid_search:
        all_metrics_grid = pd.concat([grid_function(model, model_name, df_train, df_val, seed, "Original"),
                                      grid_function(model, model_name, df_train_under, df_val_under, seed, "Under"),
                                      grid_function(model, model_name, df_train_over, df_val_over, seed, "Over")], 
                                      ignore_index=True)
        all_metrics_grid.to_csv(f"../../metrics/{model_name}_{seed}_Grid_metrics.csv", index=False)    

: 

: 

In [ ]:
repr_name="embedding_antiviral_homology_90_protT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)
seed= 42

: 

: 

In [ ]:
for model_name in MODELS.keys():
    print(f"Processing {model_name} with {repr_name}")
    main_train(df_data, seed, model_name, grid_search=True)
    print(f"Finished {model_name}")
    print("=====================================")

: 

: 